# CatBoost

Проверяю CatBoost и сравниваю с лучшим Random Forest

Лучший Random Forest: 0.9134

In [10]:
import time
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

SEED = 20260916

train = pd.read_csv("../train.csv")
validation = pd.read_csv("../validation.csv")

feature_columns = [
    column
    for column in train.columns
    if column not in ["row_id", "target"]
]

X_train = train[feature_columns]
y_train = train["target"]

X_validation = validation[feature_columns]
y_validation = validation["target"]

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Features:", len(feature_columns))

Train: (2520, 512)
Validation: (560, 512)
Features: 512


In [11]:
def evaluate_model(name, model):
    start = time.perf_counter()

    model.fit(X_train, y_train)

    probability = model.predict_proba(X_validation)[:, 1]
    auc = roc_auc_score(y_validation, probability)

    elapsed = time.perf_counter() - start

    return {
        "model": name,
        "roc_auc": auc,
        "time_sec": elapsed,
    }

In [12]:
catboost_baseline = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=SEED,
    verbose=False,
)

baseline_result = evaluate_model(
    "CatBoost baseline",
    catboost_baseline,
)

pd.DataFrame([baseline_result])

,model,roc_auc,time_sec
0,CatBoost baseline,0.914413,3.680987


### Вывод по baseline

Базовый CatBoost: 0.9144.

Это получше Random Forest с 0.9134

Поэтому пробую подобрать параметры CatBoost

In [13]:
catboost_depth_results = []

for depth in [4, 5, 6, 7, 8, 10]:
    model = CatBoostClassifier(
        iterations=500,
        depth=depth,
        learning_rate=0.05,
        random_seed=SEED,
        verbose=False,
    )

    result = evaluate_model(
        f"CatBoost depth={depth}",
        model,
    )

    result["depth"] = depth
    catboost_depth_results.append(result)

catboost_depth_df = pd.DataFrame(catboost_depth_results)
catboost_depth_df

,model,roc_auc,time_sec,depth
0,CatBoost depth=4,0.913253,1.574869,4
1,CatBoost depth=5,0.908329,2.350164,5
2,CatBoost depth=6,0.914413,3.509855,6
3,CatBoost depth=7,0.914362,5.788289,7
4,CatBoost depth=8,0.910702,10.113430,8
5,CatBoost depth=10,0.901352,32.717222,10


### Вывод по depth

Лучший - при `depth=6`: 0.9144

При `depth=7` похож но время больше

При большей глубине качество стало падать

Дальше `depth=6`

In [14]:
catboost_lr_results = []

for learning_rate in [0.01, 0.03, 0.05, 0.1, 0.2]:
    model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=learning_rate,
        random_seed=SEED,
        verbose=False,
    )

    result = evaluate_model(
        f"CatBoost learning_rate={learning_rate}",
        model,
    )

    result["learning_rate"] = learning_rate
    catboost_lr_results.append(result)

catboost_lr_df = pd.DataFrame(catboost_lr_results)
catboost_lr_df

,model,roc_auc,time_sec,learning_rate
0,CatBoost learning_rate=0.01,0.900077,3.407190,0.01
1,CatBoost learning_rate=0.03,0.907819,3.644503,0.03
2,CatBoost learning_rate=0.05,0.914413,3.379462,0.05
3,CatBoost learning_rate=0.1,0.913087,3.246446,0.10
4,CatBoost learning_rate=0.2,0.905714,3.267085,0.20


### Вывод по learning_rate

Лучший - при `learning_rate=0.05`: 0.9144.

При меньшем learning rate качество хуже

При `learning_rate=0.1` было близко, при `0.2` качество упало

Дальше `learning_rate=0.05`

In [15]:
catboost_iterations_results = []

for iterations in [200, 500, 800, 1000, 1500]:
    model = CatBoostClassifier(
        iterations=iterations,
        depth=6,
        learning_rate=0.05,
        random_seed=SEED,
        verbose=False,
    )

    result = evaluate_model(
        f"CatBoost iterations={iterations}",
        model,
    )

    result["iterations"] = iterations
    catboost_iterations_results.append(result)

catboost_iterations_df = pd.DataFrame(catboost_iterations_results)
catboost_iterations_df

,model,roc_auc,time_sec,iterations
0,CatBoost iterations=200,0.906773,1.428701,200
1,CatBoost iterations=500,0.914413,3.414598,500
2,CatBoost iterations=800,0.917449,5.596088,800
3,CatBoost iterations=1000,0.918214,7.207433,1000
4,CatBoost iterations=1500,0.919094,10.175131,1500


### Вывод по iterations

Проверю большее количество итераций.

In [16]:
catboost_iterations_more_results = []

for iterations in [1500, 2000, 2500, 3000]:
    model = CatBoostClassifier(
        iterations=iterations,
        depth=6,
        learning_rate=0.05,
        random_seed=SEED,
        verbose=False,
    )

    result = evaluate_model(
        f"CatBoost iterations={iterations}",
        model,
    )

    result["iterations"] = iterations
    catboost_iterations_more_results.append(result)

catboost_iterations_more_df = pd.DataFrame(
    catboost_iterations_more_results
)

catboost_iterations_more_df

,model,roc_auc,time_sec,iterations
0,CatBoost iterations=1500,0.919094,10.536077,1500
1,CatBoost iterations=2000,0.919235,13.504264,2000
2,CatBoost iterations=2500,0.918814,16.881859,2500
3,CatBoost iterations=3000,0.919031,20.556535,3000


### Вывод по iterations

Лучший - при `iterations=2000`: 0.9192

После 2000 качество перестало расти и начало колебаться

Дальше использую `iterations=2000`

In [17]:
catboost_l2_results = []

for l2_leaf_reg in [1, 3, 5, 10, 20]:
    model = CatBoostClassifier(
        iterations=2000,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=l2_leaf_reg,
        random_seed=SEED,
        verbose=False,
    )

    result = evaluate_model(
        f"CatBoost l2_leaf_reg={l2_leaf_reg}",
        model,
    )

    result["l2_leaf_reg"] = l2_leaf_reg
    catboost_l2_results.append(result)

catboost_l2_df = pd.DataFrame(catboost_l2_results)
catboost_l2_df

,model,roc_auc,time_sec,l2_leaf_reg
0,CatBoost l2_leaf_reg=1,0.912653,13.282876,1
1,CatBoost l2_leaf_reg=3,0.919235,13.357862,3
2,CatBoost l2_leaf_reg=5,0.918788,13.692306,5
3,CatBoost l2_leaf_reg=10,0.917372,13.796599,10
4,CatBoost l2_leaf_reg=20,0.918036,13.741045,20


### Вывод по l2_leaf_reg

Лучший - при `l2_leaf_reg=3`: 0.9192

При меньшей регуляризации качество снизилось

При больших значениях результат был хуже

Дальше `l2_leaf_reg=3`


In [18]:
catboost_random_strength_results = []

for random_strength in [0, 0.5, 1, 2, 5]:
    model = CatBoostClassifier(
        iterations=2000,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=3,
        random_strength=random_strength,
        random_seed=SEED,
        verbose=False,
    )

    result = evaluate_model(
        f"CatBoost random_strength={random_strength}",
        model,
    )

    result["random_strength"] = random_strength
    catboost_random_strength_results.append(result)

catboost_random_strength_df = pd.DataFrame(
    catboost_random_strength_results
)

catboost_random_strength_df

,model,roc_auc,time_sec,random_strength
0,CatBoost random_strength=0,0.916901,13.924573,0.0
1,CatBoost random_strength=0.5,0.918393,13.356187,0.5
2,CatBoost random_strength=1,0.919235,13.032703,1.0
3,CatBoost random_strength=2,0.919821,13.242360,2.0
4,CatBoost random_strength=5,0.914094,12.927632,5.0


### Вывод по random_strength

Лучший - при `random_strength=2`: 0.9198

При меньших значениях качество было немного хуже

При `random_strength=5` результат заметно снизился

Дальше `random_strength=2`

In [19]:
final_catboost = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    random_strength=2,
    random_seed=SEED,
    verbose=False,
)

final_catboost_result = evaluate_model(
    "Final CatBoost",
    final_catboost,
)

pd.DataFrame([final_catboost_result])

,model,roc_auc,time_sec
0,Final CatBoost,0.919821,13.528723


## Итог по CatBoost

Базовый CatBoost: 0.9144

После подбора вырос до 0.9198

Лучшие параметры:

- `iterations=2000`
- `depth=6`
- `learning_rate=0.05`
- `l2_leaf_reg=3`
- `random_strength=2`

CatBoost лучше Random Forest, у которого - 0.9134